# Submission 10 - Experiment 33B Winner

Final submission using the Experiment 33B Identity + Digit Decomposition pipeline.

Validation ROC-AUC: **0.945331**

This notebook trains the winning pipeline on the full training dataset and generates the Kaggle submission file.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

TRAIN_PATH = '../data/train.csv'
TEST_PATH = '../data/test.csv'
OUTPUT_PATH = '../submission_10.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (668665, 15)
Test shape: (286571, 14)


In [2]:
numeric_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

target_values = train[TARGET].astype(str).str.strip()
y = target_values.map({'No': 0, 'Yes': 1}).astype(int)

print('Training rows:', len(train))
print('Positive rate:', y.mean())

Training rows: 668665
Positive rate: 0.17464500160768098


In [3]:
# ------------------------------------------------------------
# 33B DIGIT DECOMPOSITION
# ------------------------------------------------------------

def add_digit_features(df, columns):
    out = df.copy()

    for col in columns:
        values = pd.to_numeric(out[col], errors='coerce')
        safe = values.fillna(0).abs().astype(np.int64)
        strings = safe.astype(str)

        out[f'{col}__digit_count'] = strings.str.len().astype(float)
        out[f'{col}__first_digit'] = strings.str[0].astype(float)
        out[f'{col}__last_digit'] = (safe % 10).astype(float)

        out[f'{col}__digit_sum'] = strings.apply(
            lambda x: sum(int(ch) for ch in x)
        ).astype(float)

        out[f'{col}__parity'] = (safe % 2).astype(float)
        out[f'{col}__mod100'] = (safe % 100).astype(float)
        out[f'{col}__mod1000'] = (safe % 1000).astype(float)
        out[f'{col}__ends_zero'] = (safe % 10 == 0).astype(float)

        # Preserve missingness in the derived features.
        missing = values.isna()
        digit_cols = [
            f'{col}__digit_count',
            f'{col}__first_digit',
            f'{col}__last_digit',
            f'{col}__digit_sum',
            f'{col}__parity',
            f'{col}__mod100',
            f'{col}__mod1000',
            f'{col}__ends_zero'
        ]
        out.loc[missing, digit_cols] = np.nan

    return out

train_features = train.drop(columns=[TARGET, ID_COL]).copy()
test_features = test.drop(columns=[ID_COL]).copy()

train_features = add_digit_features(train_features, numeric_cols)
test_features = add_digit_features(test_features, numeric_cols)

digit_cols = []
for col in numeric_cols:
    digit_cols.extend([
        f'{col}__digit_count',
        f'{col}__first_digit',
        f'{col}__last_digit',
        f'{col}__digit_sum',
        f'{col}__parity',
        f'{col}__mod100',
        f'{col}__mod1000',
        f'{col}__ends_zero'
    ])

print('Digit features:', len(digit_cols))
print('Total feature columns before encoding:', train_features.shape[1])

Digit features: 56
Total feature columns before encoding: 69


In [4]:
# ------------------------------------------------------------
# IDENTITY TARGET ENCODING + FREQUENCY
# Same identity representation used by the 23B/33B pipeline.
# ------------------------------------------------------------

def identity_key(series):
    return series.astype('string').fillna('__MISSING__')

def smoothed_target_map(series, target, smoothing=20):
    key = identity_key(series)
    tmp = pd.DataFrame({'key': key, 'target': target.values})
    stats = tmp.groupby('key')['target'].agg(['count', 'mean'])
    global_mean = float(target.mean())
    stats['encoded'] = (
        (stats['count'] * stats['mean']) + (smoothing * global_mean)
    ) / (stats['count'] + smoothing)
    return stats['encoded'], global_mean

def apply_target_mapping(series, mapping, global_mean):
    key = identity_key(series)
    return key.map(mapping).fillna(global_mean).astype(float)

def frequency_map(series):
    key = identity_key(series)
    return key.value_counts(normalize=True)

def apply_frequency_mapping(series, mapping):
    key = identity_key(series)
    return key.map(mapping).fillna(0.0).astype(float)

SMOOTHING = 20
N_SPLITS = 3

# OOF target encoding for the full training data.
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for col in numeric_cols:
    encoded = np.zeros(len(train_features), dtype=float)

    for train_idx, valid_idx in skf.split(train_features, y):
        mapping, global_mean = smoothed_target_map(
            train_features.iloc[train_idx][col],
            y.iloc[train_idx],
            smoothing=SMOOTHING
        )

        encoded[valid_idx] = apply_target_mapping(
            train_features.iloc[valid_idx][col],
            mapping,
            global_mean
        ).values

    train_features[f'{col}__identity_target'] = encoded

    # Frequency encoding fitted on the complete training data.
    freq = frequency_map(train_features[col])
    train_features[f'{col}__identity_frequency'] = apply_frequency_mapping(
        train_features[col], freq
    )

    # Fit the final target/frequency mappings on all training data for test.
    full_mapping, full_global_mean = smoothed_target_map(
        train_features[col], y, smoothing=SMOOTHING
    )

    test_features[f'{col}__identity_target'] = apply_target_mapping(
        test_features[col], full_mapping, full_global_mean
    )

    test_features[f'{col}__identity_frequency'] = apply_frequency_mapping(
        test_features[col], freq
    )

print('Final training feature count:', train_features.shape[1])
print('Final test feature count:', test_features.shape[1])

Final training feature count: 83
Final test feature count: 83


In [5]:
# ------------------------------------------------------------
# FINAL MODEL
# Same XGBoost configuration as the 23B/33B benchmark.
# ------------------------------------------------------------

numeric_final = numeric_cols + digit_cols + [
    f'{col}__identity_target' for col in numeric_cols
] + [
    f'{col}__identity_frequency' for col in numeric_cols
]

categorical_final = categorical_cols

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median'))
            ]),
            numeric_final
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_final
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(train_features)
X_test_encoded = preprocessor.transform(test_features)

print('Encoded train shape:', X_train_encoded.shape)
print('Encoded test shape:', X_test_encoded.shape)

model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_encoded, y)

test_predictions = model.predict_proba(X_test_encoded)[:, 1]

print('Prediction count:', len(test_predictions))
print('Prediction range:', test_predictions.min(), test_predictions.max())

Encoded train shape: (668665, 94)
Encoded test shape: (286571, 94)
Prediction count: 286571
Prediction range: 2.9641956e-06 0.9997428


In [6]:
# ------------------------------------------------------------
# SUBMISSION 10
# ------------------------------------------------------------

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_predictions
})

submission.to_csv(OUTPUT_PATH, index=False)

print('Submission saved to:', OUTPUT_PATH)
print('Submission shape:', submission.shape)
print(submission.head())
print('\nColumns:', submission.columns.tolist())

Submission saved to: ../submission_10.csv
Submission shape: (286571, 2)
       id  Will_Buy_EV
0  668665     0.039391
1  668666     0.013093
2  668667     0.003532
3  668668     0.002304
4  668669     0.027263

Columns: ['id', 'Will_Buy_EV']
